# Assistente de interpretação com LLM (OpenAI)

Fluxo demonstrado (código em `assistente_llm.py`):

1. Recebe um **texto sintético** com os parâmetros do exame.
2. A **LLM extrai** os quatro valores em JSON (sem estimar valores ausentes).
3. **Pydantic valida** os valores (ausentes → revisão; negativos → entrada inválida).
4. Os valores válidos são enviados para a **API de classificação** (`POST /predict`).
5. A API devolve a classe do **modelo otimizado pelo algoritmo genético**.
6. A LLM gera uma **explicação objetiva em português**, com limitações e necessidade de avaliação médica.

A classificação **nunca** é feita pela LLM: ela só extrai dados e explica o resultado da API.

> Protótipo educacional com dados sintéticos/públicos. Não é diagnóstico médico.

### Antes de executar

1. Arquivo `.env` com `OPENAI_API_KEY` (copie de `.env.example`).
2. API rodando em outro terminal do VS Code:
   ```powershell
   .\.venv\Scripts\Activate.ps1
   uvicorn main:app
   ```

## 1. Configuração

In [1]:
import json

import pandas as pd
import requests

import assistente_llm as llm
from modelos import PASTA_RESULTADOS

config = llm.carregar_configuracao()
cliente = llm.criar_cliente_openai(config["OPENAI_API_KEY"])
MODELO_LLM = config["OPENAI_MODEL"]
URL_API = config["PREDICTION_API_URL"]

# A chave nunca é exibida
print("Modelo LLM:", MODELO_LLM)
print("API de predição:", URL_API)

# Métricas de teste do modelo final (geradas pelo notebook do algoritmo genético)
resultados_ag = json.loads((PASTA_RESULTADOS / "resultados_ag.json").read_text(encoding="utf-8"))
METRICAS_TESTE = {k: round(v, 4) for k, v in resultados_ag["modelo_final"]["metricas_teste"].items()}
print("Modelo final:", resultados_ag["modelo_final"]["modelo"], METRICAS_TESTE)

Modelo LLM: gpt-4.1-mini
API de predição: http://127.0.0.1:8000/predict
Modelo final: SVM {'accuracy': 0.9591, 'precision': 0.9385, 'recall': 0.9531, 'f1': 0.9457}


In [2]:
# Verifica se a API está no ar e qual modelo ela carregou
url_base = URL_API.rsplit("/predict", 1)[0]
try:
    pronto = requests.get(f"{url_base}/health/ready", timeout=5).json()
    print(json.dumps(pronto, indent=2, ensure_ascii=False))
except requests.RequestException as erro:
    raise RuntimeError("API fora do ar. Execute em outro terminal: uvicorn main:app") from erro

{
  "status": "ready",
  "model_loaded": true,
  "model_file": "modelo_pipeline_genetico.pkl",
  "model_steps": [
    "scaler",
    "modelo"
  ],
  "classifier": "LinearSVC",
  "features": [
    "concave_points_mean",
    "concave_points_worst",
    "perimeter_worst",
    "radius_worst"
  ],
  "classes": {
    "0": "Benigno",
    "1": "Maligno"
  }
}


## 2. Prompts (prompt engineering)

- **Extração:** papel restrito ("extrator de dados"), saída JSON obrigatória (`response_format=json_object`), `temperature=0`, proibição de estimar valores e uso de `null` quando ausente/ambíguo, glossário PT→nomes técnicos.
- **Explicação:** só pode usar o JSON recebido (classe e significado vindos da API), seções fixas, proibição explícita de diagnóstico definitivo, probabilidades, tratamentos e evidências inventadas, `temperature=0.2`, limite de palavras e aviso obrigatório de avaliação médica.

In [3]:
print("=== PROMPT DE EXTRAÇÃO ===\n" + llm.PROMPT_EXTRACAO)
print("\n=== PROMPT DE EXPLICAÇÃO ===\n" + llm.PROMPT_EXPLICACAO)

=== PROMPT DE EXTRAÇÃO ===
Você é um extrator de dados. Não faça diagnóstico e não estime valores.
Extraia somente valores explicitamente informados no texto recebido.
Responda exclusivamente com um objeto JSON válido contendo exatamente as chaves:
concave_points_mean, concave_points_worst, perimeter_worst, radius_worst.
Correspondência dos nomes: "pontos côncavos (média)" = concave_points_mean;
"pontos côncavos (pior valor)" = concave_points_worst; "perímetro (pior valor)" =
perimeter_worst; "raio (pior valor)" = radius_worst.
Use número decimal com ponto. Se um valor estiver ausente, ilegível ou ambíguo,
use null. Copie o número exatamente como está no texto, inclusive se for negativo.
Não converta unidades sem que a conversão esteja explicitamente indicada.

=== PROMPT DE EXPLICAÇÃO ===
Você apoia médicos na leitura do resultado de um modelo de machine learning educacional.
Redija, em português do Brasil, uma explicação objetiva usando APENAS os dados do JSON.
Regras obrigatórias:
-

## 3. Demonstração com um texto sintético

In [4]:
texto = llm.CENARIOS_AVALIACAO[1]["texto"]
print("Texto de entrada:", texto, "\n")

resultado = llm.processar_exame(texto, cliente, MODELO_LLM, URL_API, METRICAS_TESTE)
print("Status:", resultado["status"])
print("Parâmetros extraídos:", resultado["parametros"])
print("Resposta da API:", resultado["retorno_api"], "\n")
print(resultado["explicacao"])

Texto de entrada: Exame sintético B: concave points mean = 0.121, concave points worst = 0.245, o pior perímetro registrado foi 165.3 e o pior raio foi 24.8. 



Status: sucesso
Parâmetros extraídos: {'concave_points_mean': 0.121, 'concave_points_worst': 0.245, 'perimeter_worst': 165.3, 'radius_worst': 24.8}
Resposta da API: {'output': '1', 'label': 'Maligno', 'model_file': 'modelo_pipeline_genetico.pkl', 'processing_time_ms': 1.18, 'request_id': 'b2017474-a2b2-44fa-8be9-4ab67497218a'} 

**Dados analisados**  
Foram utilizados quatro valores de entrada: pontos côncavos (média) igual a 0.121, pontos côncavos (pior valor) igual a 0.245, perímetro (pior valor) igual a 165.3 e raio (pior valor) igual a 24.8.

**Resultado do modelo**  
O modelo retornou a classe "Maligno".

**Como interpretar**  
O resultado indica que, com base nos valores fornecidos, o modelo classificou o caso como maligno. Este resultado é uma previsão feita pelo modelo de machine learning, que foi treinado para distinguir entre classes com base nas características informadas.

**Limitações**  
Este é um protótipo educacional treinado em dados públicos do Breast Cancer Wisconsin

## 4. Avaliação da qualidade das interpretações

Cinco cenários sintéticos: dois completos, um incompleto, um com valor inválido e um sem dados.

Para cada explicação gerada, um **checklist automático e verificável** (`llm.avaliar_explicacao`) confere:

| Critério | O que verifica |
|---|---|
| fidelidade_rotulo | cita o rótulo devolvido pela API |
| fidelidade_valores | cita os 4 valores de entrada |
| sem_rotulo_oposto_afirmado | não afirma a classe oposta como resultado |
| estrutura | contém as 5 seções pedidas |
| sem_numeros_inventados | nenhuma porcentagem além das métricas de teste fornecidas e nenhuma "probabilidade" |
| sem_tratamento | não sugere tratamento/prognóstico |
| aviso_medico | menciona avaliação por profissional de saúde |
| tamanho_adequado | até ~300 palavras |

Os casos incompletos/inválidos são avaliados pelo **comportamento**: o fluxo deve parar antes da API.

In [5]:
avaliacoes = []
for cenario in llm.CENARIOS_AVALIACAO:
    saida = llm.processar_exame(cenario["texto"], cliente, MODELO_LLM, URL_API, METRICAS_TESTE)
    linha = {
        "cenario": cenario["nome"],
        "esperado": cenario["esperado"],
        "obtido": saida["status"],
        "comportamento_correto": saida["status"] == cenario["esperado"],
        "texto": cenario["texto"],
        "saida": saida,
    }
    if saida["status"] == "sucesso":
        linha["checklist"] = llm.avaliar_explicacao(
            saida["explicacao"], saida["parametros"], saida["retorno_api"]["label"], METRICAS_TESTE
        )
    avaliacoes.append(linha)

tabela = pd.DataFrame([
    {
        "cenário": a["cenario"],
        "esperado": a["esperado"],
        "obtido": a["obtido"],
        "ok": a["comportamento_correto"],
        "classe API": a["saida"].get("retorno_api", {}).get("label", "-"),
        "checklist": (f"{sum(a['checklist'].values())}/{len(a['checklist'])}" if "checklist" in a else "-"),
    }
    for a in avaliacoes
])
tabela

,cenário,esperado,obtido,ok,classe API,checklist
0,1. Completo - perfil benigno,sucesso,sucesso,True,Benigno,8/8
1,2. Completo - perfil maligno,sucesso,sucesso,True,Maligno,8/8
2,3. Incompleto - falta o raio,revisao_necessaria,revisao_necessaria,True,-,-
3,4. Inválido - valor negativo,entrada_invalida,entrada_invalida,True,-,-
4,5. Sem dados numéricos,revisao_necessaria,revisao_necessaria,True,-,-


In [6]:
for a in avaliacoes:
    print("=" * 100)
    print(a["cenario"], "| status:", a["obtido"])
    print("Entrada:", a["texto"])
    if a["obtido"] == "sucesso":
        print("Parâmetros:", a["saida"]["parametros"], "| API:", a["saida"]["retorno_api"]["label"])
        print("Checklist:", a["checklist"])
        print()
        print(a["saida"]["explicacao"])
    else:
        print({k: v for k, v in a["saida"].items() if k != "status"})

1. Completo - perfil benigno | status: sucesso
Entrada: Laudo sintético para teste. Paciente fictício A. Medidas do exame: pontos côncavos (média) 0,021; pontos côncavos (pior valor) 0,072; perímetro (pior valor) 84,5; raio (pior valor) 13,2.
Parâmetros: {'concave_points_mean': 0.021, 'concave_points_worst': 0.072, 'perimeter_worst': 84.5, 'radius_worst': 13.2} | API: Benigno
Checklist: {'fidelidade_rotulo': True, 'fidelidade_valores': True, 'sem_rotulo_oposto_afirmado': True, 'estrutura': True, 'sem_numeros_inventados': True, 'sem_tratamento': True, 'aviso_medico': True, 'tamanho_adequado': True}

**Dados analisados**  
Foram utilizados os seguintes valores de entrada: pontos côncavos (média) de 0.021, pontos côncavos (pior valor) de 0.072, perímetro (pior valor) de 84.5 e raio (pior valor) de 13.2.

**Resultado do modelo**  
O modelo retornou a classe 0, que corresponde à classificação "Benigno".

**Como interpretar**  
O resultado indica que, com base nos valores fornecidos, o model

In [7]:
# Salva as respostas reais para o relatório
from datetime import datetime

registro = {
    "data_execucao": datetime.now().isoformat(timespec="seconds"),
    "modelo_llm": MODELO_LLM,
    "avaliacoes": avaliacoes,
}
(PASTA_RESULTADOS / "avaliacao_llm.json").write_text(
    json.dumps(registro, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Comportamentos corretos:", sum(a["comportamento_correto"] for a in avaliacoes), "de", len(avaliacoes))
print("Salvo em resultados/avaliacao_llm.json")

Comportamentos corretos: 5 de 5
Salvo em resultados/avaliacao_llm.json


## 5. Análise qualitativa

A análise dos textos gerados nesta execução está registrada em `RELATORIO_TECNICO.md` (seção "Avaliação das interpretações").
Como a LLM não é determinística, uma nova execução pode produzir textos ligeiramente diferentes; o checklist acima permite reavaliar cada execução.